# Phase 2 — SFT Retrain + Baseline Comparison

Two runs, identical config. Only variable: invalid class distribution.

| Run | Dataset | WandB name |
|-----|---------|------------|
| 1 | `data/dataset_final_qwen.jsonl` (curated, balanced) | `curated-3ep` |
| 2 | `data/dataset_baseline_qwen.jsonl` (random invalids) | `baseline-3ep` |

**Before running:**
1. Copy `dataset_syzkaller_*.jsonl.gz` into `data/corpus/`
2. Copy `dataset_final_qwen.jsonl` into `data/`
3. Build baseline: `python ml/build_baseline_dataset.py`
4. Copy checkpoint into `checkpoints/sft_fase1/` (must contain `checkpoint-1500/`)
5. Run `wandb login` once in terminal

In [ ]:
# ── CONFIG ── set these once, then run all cells ──────────────────────────────
import subprocess
from pathlib import Path

REPO_ROOT = Path(subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"], text=True
).strip())

CURATED_DATASET  = REPO_ROOT / "data" / "dataset_final_qwen.jsonl"
BASELINE_DATASET = REPO_ROOT / "data" / "dataset_baseline_qwen.jsonl"

# Checkpoint from Phase 1 SFT (copy into checkpoints/sft_fase1/)
CHECKPOINT_PATH  = REPO_ROOT / "checkpoints" / "sft_fase1" / "checkpoint-1500"

CURATED_OUTPUT   = REPO_ROOT / "checkpoints" / "curated_3ep"
BASELINE_OUTPUT  = REPO_ROOT / "checkpoints" / "baseline_3ep"

# Sanity check
for p, label in [
    (CURATED_DATASET,  "curated dataset"),
    (BASELINE_DATASET, "baseline dataset"),
    (CHECKPOINT_PATH,  "Phase 1 checkpoint"),
]:
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}] {label}: {p}")

In [ ]:
import torch
import wandb
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B"

def load_tokenizer():
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

def build_prompt(sample, tokenizer):
    kernel_ver = sample.get("kernel_version", "unknown")
    assembly   = sample.get("verifier_log", "")
    if sample.get("is_valid", False):
        text = f"Kernel: {kernel_ver} | Status: VALID\n### ASSEMBLY:\n{assembly}{tokenizer.eos_token}"
    else:
        err  = sample.get("error_reason_clean", "Unknown error")
        text = f"Kernel: {kernel_ver} | Status: INVALID | Error: {err}\n### ASSEMBLY:\n{assembly}{tokenizer.eos_token}"
    return {"formatted_prompt": text}

def tokenize(sample, tokenizer):
    enc = tokenizer(
        sample["formatted_prompt"],
        max_length=768,
        truncation=True,
        padding="max_length",
    )
    enc["labels"] = [
        l if l != tokenizer.pad_token_id else -100
        for l in enc["input_ids"]
    ]
    return enc

def load_model_fresh():
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb,
        device_map="auto",
        attn_implementation="sdpa",
    )
    model.gradient_checkpointing_enable()
    lora = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    )
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()
    return model

def make_training_args(output_dir, wandb_name, resume_from=None):
    return TrainingArguments(
        output_dir=str(output_dir),
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        bf16=True,
        optim="paged_adamw_8bit",
        logging_steps=10,
        save_steps=200,
        eval_strategy="steps",
        eval_steps=200,
        load_best_model_at_end=True,
        report_to="wandb",
        run_name=wandb_name,
    )

print("[*] Shared setup loaded.")

## Run 1 — Curated model
Resumes from `checkpoints/sft_fase1/checkpoint-1500` (Phase 1 SFT).
Run this cell, wait for it to finish, then run Run 2.

In [ ]:
WANDB_NAME = "curated-3ep"
wandb.init(project="ebpf-thesis", name=WANDB_NAME, reinit=True)

tokenizer = load_tokenizer()

dataset = load_dataset("json", data_files=str(CURATED_DATASET), split="train")
split   = dataset.train_test_split(test_size=0.1, seed=42)
train   = split["train"].map(lambda s: build_prompt(s, tokenizer)).map(lambda s: tokenize(s, tokenizer))
val     = split["test"].map(lambda s: build_prompt(s, tokenizer)).map(lambda s: tokenize(s, tokenizer))

print(f"[*] Train: {len(train)}  Val: {len(val)}")

# Load base + resume LoRA weights from Phase 1 checkpoint
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
base  = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map="auto", attn_implementation="sdpa"
)
base.gradient_checkpointing_enable()
model = PeftModel.from_pretrained(base, str(CHECKPOINT_PATH), is_trainable=True)
model.print_trainable_parameters()

trainer = Trainer(
    model=model,
    args=make_training_args(CURATED_OUTPUT, WANDB_NAME),
    train_dataset=train,
    eval_dataset=val,
)

trainer.train()
trainer.save_model(str(CURATED_OUTPUT / "adattatore_ebpf_v1"))
tokenizer.save_pretrained(str(CURATED_OUTPUT / "adattatore_ebpf_v1"))
wandb.finish()
print(f"[+] Curated model saved → {CURATED_OUTPUT / 'adattatore_ebpf_v1'}")

## Run 2 — Baseline model
Trains from scratch (fresh LoRA). Build baseline first if not done:
```
python ml/build_baseline_dataset.py
```

In [ ]:
WANDB_NAME = "baseline-3ep"
wandb.init(project="ebpf-thesis", name=WANDB_NAME, reinit=True)

tokenizer = load_tokenizer()

dataset = load_dataset("json", data_files=str(BASELINE_DATASET), split="train")
split   = dataset.train_test_split(test_size=0.1, seed=42)
train   = split["train"].map(lambda s: build_prompt(s, tokenizer)).map(lambda s: tokenize(s, tokenizer))
val     = split["test"].map(lambda s: build_prompt(s, tokenizer)).map(lambda s: tokenize(s, tokenizer))

print(f"[*] Train: {len(train)}  Val: {len(val)}")

model   = load_model_fresh()
trainer = Trainer(
    model=model,
    args=make_training_args(BASELINE_OUTPUT, WANDB_NAME),
    train_dataset=train,
    eval_dataset=val,
)

trainer.train()
trainer.save_model(str(BASELINE_OUTPUT / "adattatore_ebpf_v1"))
tokenizer.save_pretrained(str(BASELINE_OUTPUT / "adattatore_ebpf_v1"))
wandb.finish()
print(f"[+] Baseline model saved → {BASELINE_OUTPUT / 'adattatore_ebpf_v1'}")